In [1]:
import os
import re
import polars as pl
from tqdm import tqdm

from typing import List, Dict, Any

from llm_benchmark.utils.enums import QuestionType
from llm_benchmark.data.eval import get_ids_from_row, get_actual_row
from llm_benchmark.utils.dataset import Dataset, DatasetModule
from llm_benchmark import config as cfg

import traceback

from llm_benchmark.utils.benchmark import seshat_setup

dataset: Dataset = seshat_setup(
    seshat_cache_dir="/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat", 
    force=False
    )

def parse_csv_to_df(dataset: Dataset, model_name: str, model_database_source: str, category: str,
                    path_to_types: str, question_type: QuestionType,) -> List[pl.DataFrame]:
    try:
        csv_paths: List[str] = os.listdir(os.path.join(path_to_types, question_type.name.lower()))
    except:
        return []
    dfs: List[pl.DataFrame] = []
    
    for csv_path in tqdm(csv_paths, desc="     Loading csvs"):
        csv: pl.DataFrame = pl.read_csv(os.path.join(path_to_types, question_type.name.lower(), csv_path))
        new_dicts: List[Dict[str, Any]] = []

        endpoint_identifier: str = csv[1, 0].replace(cfg.ENDPOINT_URL, "")[:-1]
        dataset_module: DatasetModule = dataset.get_module(identifier=endpoint_identifier.replace(cfg.ENDPOINT_URL, "",))
        dataset_df: pl.DataFrame = dataset_module.get_entries()

        for row in csv.iter_rows(named=True):
            try:
                output_split: List[str] = re.findall(r'(?:ANSWER|REASONING):\s*(.*)', row["output"])
                reasoning: str = output_split[0]
                answer: str = output_split[1]
            except:
                reasoning: str = ""
                answer: str = "failed"

            entry_idx: int = int(row["entry_idx"])
            row_actual: Dict[str, Any] = get_actual_row(dataset_df=dataset_df, entry_idx=entry_idx)

            try:
                ids: Dict[str, Any] = get_ids_from_row(dataset.grouping, row_actual, endpoint_identifier)
            except Exception as e:
                print(f"ID Error: {type(e).__name__}: {e}")
                print(f"Endpoint: {endpoint_identifier}")
                print(f"Row keys: {list(row_actual.keys())}")
                print(traceback.format_exc())
                ids: Dict[str, Any] = {}

            entry: Dict[str, Any] = {
                "endpoint_identifier": row["endpoint_identifier"],
                "entry_idx": entry_idx,
                "model": row["model"],
                "db_model": model_database_source,
                "reasoning": reasoning,
                "answer": answer,
                "message": row["message"],
            }
            entry.update(ids)
            new_dicts.append(entry)
            
        dfs.append(pl.DataFrame(new_dicts))
    return dfs
        
            

def parse_all(path: str, question_type: QuestionType, dataset: Dataset) -> pl.DataFrame:
    df: pl.DataFrame = pl.DataFrame({})
    for company in os.listdir(path):
        model_path: str = os.path.join(path, company)
        models_per_db: List[str] = os.listdir(model_path) 
        for model in tqdm(models_per_db, desc=f"Loading model: {company}"):
            for model_version in os.listdir(os.path.join(model_path, model)):
                model_per_db_path: str = os.path.join(model_path, model, model_version)
                itms: List[str] = model_version.split('+')

                assert ('+' in model_version)

                model_name: str = itms[0]
                model_database_src: str = itms[1]

                for category in os.listdir(model_per_db_path):
                    path_to_types: str = os.path.join(model_per_db_path, category)
                    if category == ".DS_Store":
                        continue
                    
                    cat_dfs: List[pl.DataFrame] = parse_csv_to_df(
                        dataset=dataset,
                        model_name=model_name,
                        model_database_source=model_database_src,
                        category=category,
                        question_type=question_type,
                        path_to_types=path_to_types
                    ) 

                    try:
                        df = pl.concat([df, *cat_dfs], how="diagonal_relaxed")
                    except:
                        print("Failed to concat df")
    
    return df



/Users/apple/miniconda3/envs/llm_benchmark/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Ignoring polity core/macro-regions as per configuration.
core/regions https://seshat-db.com/api/core/regions/
Ignoring polity core/regions as per configuration.
core/ngas https://seshat-db.com/api/core/ngas/
Ignoring polity core/ngas as per configuration.
core/polities https://seshat-db.com/api/core/polities/
Ignoring polity core/polities as per configuration.
core/capitals https://seshat-db.com/api/core/capitals/
Ignoring polity core/capitals as per configuration.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Ignoring polity core/nga-polity-relations as per configuration.
core/sections https://seshat-db.com/api/core/sections/
Ignoring polity core/sections as per configuration.
core/subsections https://seshat-db.com/api/core/subsections/
Ignoring polity core/subsections as per configuration.
core/variable-hierarchies https://seshat-db.com/api/core/variable

In [2]:

eval_src: str = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/eval/final"

df: pl.DataFrame = parse_all(
    path=eval_src,
    question_type=QuestionType.MULTIPLE_CHOICE,
    dataset=dataset
    )

Loading model: gemini:   0%|          | 0/4 [00:00<?, ?it/s]

     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 100%|██████████| 60/60 [00:00<00:00, 201.09it/s]


     Loading csvs: 100%|██████████| 60/60 [00:00<00:00, 195.14it/s]]


     Loading csvs: 100%|██████████| 60/60 [00:00<00:00, 177.94it/s]]


     Loading csvs: 100%|██████████| 20/20 [00:00<00:00, 382.56it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 100%|██████████| 20/20 [00:00<00:00, 380.05it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 100%|██████████| 43/43 [00:00<00:00, 160.55it/s]
     Loading csvs: 0it [00:00, ?it/s]2/5 [00:00<00:01,  2.20it/s]
     Loading csvs: 100%|██████████| 22/22 [00:00<00:00, 359.25it/s]


     Loading csvs: 100%|██████████| 22/22 [00:00<00:00, 319.69it/s]


     Loading csvs: 100%|██████████| 22/22 [00:00<00:00, 161.37it/s]


     Loading csvs: 100%|██████████| 77/77 [00:00<00:00, 161.91it/s]



     Loading csvs:  64%|██████▎   | 7/11 [00:00<00:00,  8.32it/s]

     Loading csvs: 100%|██████████| 49/49 [00:00<00:00, 181.77it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
     Loading csvs: 0it [00:00, ?it/s]
Loading model: gpt: 100%|██████████| 5/5 [00:44<00:00,  8.95s/it]


In [5]:
def group(
        df: pl.DataFrame,
        metric: str,
        ) -> pl.DataFrame:
    return df\
    .select(["llm_model", "endpoint", "result", metric])\
    .group_by(["llm_model", "endpoint", metric])\
    .agg(
        pl.col("result").mean().alias("per_metric")
    )\
    .group_by(["llm_model", metric])\
    .agg(
        pl.format(
            "{} [{}, {}]",
            (pl.col("per_metric").mean() * 100).round(1),
            (pl.col("per_metric").min() * 100).round(1),
            (pl.col("per_metric").max() * 100).round(1)
            ).alias("metrics")
    )\
    .pivot(
        on="llm_model",
        index=metric,
        values="metrics"
    )\
    .sort([metric])

In [11]:
metric = "macro_str"

with pl.Config(set_tbl_rows=500):
    print(
        df
        .select(["model", "endpoint_identifier", "answer", metric])
        .group_by(["model", "endpoint_identifier", metric, "answer"])
        .agg(
            pl.len().alias("per_metric")
        )
        .group_by(["model", metric])
        .head(500)
    )

shape: (6_529, 5)
┌─────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────┐
│ model               ┆ macro_str           ┆ endpoint_identifie ┆ answer             ┆ per_metric │
│ ---                 ┆ ---                 ┆ r                  ┆ ---                ┆ ---        │
│ str                 ┆ str                 ┆ ---                ┆ str                ┆ u32        │
│                     ┆                     ┆ str                ┆                    ┆            │
╞═════════════════════╪═════════════════════╪════════════════════╪════════════════════╪════════════╡
│ gemini-2.5-flash    ┆ East Asia           ┆ https://seshat-db. ┆ A                  ┆ 14         │
│                     ┆                     ┆ com/api/rt/w…      ┆                    ┆            │
│ gemini-2.5-flash    ┆ East Asia           ┆ https://seshat-db. ┆ A                  ┆ 10         │
│                     ┆                     ┆ com/api/ec/l…      ┆       

In [ ]:
# Filter down to Qwen-7B-Chat where the endpoint score is exactly 0
zero_endpoints = (
    df
    .filter(pl.col("llm_model") == "Qwen-7B-Chat")
    .group_by(["endpoint", metric])
    .agg(pl.col("result").mean().alias("per_metric"))
    .filter(pl.col("per_metric") == 0)
)

print(zero_endpoints)

In [ ]:
print(df.columns)
print(df['section_str'].unique().sort())